In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
import yaml
from MPCModelFuncs import differential_mpc_model
from MPCModelFuncs import euler_step

print(tf.__version__)
# physical_devices = tf.config.list_physical_devices('GPU')
# tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)

# Data directory
dir = '/home/david/fst/autonomous-systems/src/control/learning_mpcc/AI/train_data/cm_clean_T1/'
modelName = 'model_17'
T = 1
h = 0.050

x_test, states, states_next = [], [], []
# Load data
npz_files = [f for f in os.listdir(dir) if f.endswith(".npz")]
for file in npz_files:
    data = np.load(dir + file)
    x_test.append(data['x_test'])
    states.append(data['states_test'])
    states_next.append(data['states_next_test'])
x_test = np.concatenate(x_test, axis=0)
states = np.concatenate(states, axis=0)
states_next = np.concatenate(states_next, axis=0)

# Read yaml file
p = np.zeros(22)
with open('/home/david/fst/autonomous-systems/src/control/learning_mpcc/config/default.yaml') as stream:
    parameters = yaml.load(stream, Loader=yaml.SafeLoader)
    p[0] = parameters['model_params']['l_f'];
    p[1] = parameters['model_params']['l_r'];
    p[2] = parameters['model_params']['m'];
    p[3] = parameters['model_params']['I_z'];
    p[4] = parameters['model_params']['T_max_front'];
    p[5] = parameters['model_params']['T_max_rear'];
    p[6] = parameters['model_params']['T_brake_front'];
    p[7] = parameters['model_params']['T_brake_rear'];
    p[8] = parameters['model_params']['GR'];
    p[9] = parameters['model_params']['eta_motor'];
    p[10] = parameters['model_params']['r_wheel'];
    p[11] = parameters['model_params']['g'];
    p[12] = parameters['model_params']['C_roll'];
    p[13] = parameters['model_params']['rho'];
    p[14] = parameters['model_params']['C_d'];
    p[15] = parameters['model_params']['C_l'];
    p[16] = parameters['tyre_params']['B'];
    p[17] = parameters['tyre_params']['C'];
    p[18] = parameters['tyre_params']['D'];
    p[19] = parameters['model_params']['downforce_front'];
    p[20] = parameters['model_params']['downforce_rear'];
    p[21] = parameters['model_params']['h_cog'];

In [ ]:
model = tf.keras.models.load_model('saved_models/' + modelName + '_T_' + str(T) + '.h5')

# Check its architecture
model.summary()

In [ ]:
control_actions = np.array([x_test[:,-1,0], x_test[:,-1,1], x_test[:,-2,0], x_test[:,-2,1]]).T
dX = differential_mpc_model(states, control_actions, p).T
NN_correction = model.predict(x_test)
states_next_model = euler_step(states, dX, h)
states_next_NN = euler_step(states, dX+NN_correction, h)
 

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(100, 100))

ax[0].plot(states_next[:,0], label='True',linewidth=1)
ax[0].plot(states_next_model[:,0], label='Model Predicted',linewidth=1.5, linestyle='--')
ax[0].plot(states_next_NN[:,0], label='NN Predicted',linewidth=1.5, linestyle=':')
ax[0].set_xlabel('Timestep',fontsize=60)
ax[0].set_ylabel('Velocity x [m/s]',fontsize=60)
ax[0].set_title('vx',fontsize=60)
ax[0].legend(fontsize=60)

ax[1].plot(states_next[:,1], label='True',linewidth=1)
ax[1].plot(states_next_model[:,1], label='Model Predicted',linewidth=1.5, linestyle='--')
ax[1].plot(states_next_NN[:,1], label='NN Predicted',linewidth=1.5, linestyle=':')
ax[1].set_xlabel('Timestep',fontsize=60)
ax[1].set_ylabel('Velocity y [m/s]',fontsize=60)
ax[1].set_title('vy',fontsize=60)
ax[1].legend(fontsize=60)

ax[2].plot(states_next[:,2], label='True',linewidth=1)
ax[2].plot(states_next_model[:,2], label='Model Predicted',linewidth=1.5, linestyle='--')
ax[2].plot(states_next_NN[:,2], label='NN Predicted',linewidth=1.5, linestyle=':')
ax[2].set_xlabel('Timestep',fontsize=60)
ax[2].set_ylabel('Yaw Rate [rad/s]',fontsize=60)
ax[2].set_title('yaw rate',fontsize=60)
ax[2].legend(fontsize=60)

plt.show()

In [ ]:
RMSE_vx_model = np.sqrt(np.mean((states_next[:,0] - states_next_model[:,0])**2))
RMSE_vy_model = np.sqrt(np.mean((states_next[:,1] - states_next_model[:,1])**2))
RMSE_r_model = np.sqrt(np.mean((states_next[:,2]- states_next_model[:,2])**2))

RMSE_vx_NN = np.sqrt(np.mean((states_next[:,0] - states_next_NN[:,0])**2))
RMSE_vy_NN = np.sqrt(np.mean((states_next[:,1] - states_next_NN[:,1])**2))
RMSE_r_NN = np.sqrt(np.mean((states_next[:,2] - states_next_NN[:,2])**2))

max_vx_model = np.max(np.abs(states_next[:,0] - states_next_model[:,0]))
max_vy_model = np.max(np.abs(states_next[:,1] - states_next_model[:,1]))
max_r_model = np.max(np.abs(states_next[:,2] - states_next_model[:,2]))

max_vx_NN = np.max(np.abs(states_next[:,0] - states_next_NN[:,0]))
max_vy_NN = np.max(np.abs(states_next[:,1] - states_next_NN[:,1]))
max_r_NN = np.max(np.abs(states_next[:,2] - states_next_NN[:,2]))

print('RMSE of the true and predicted states:')
print("Model - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_model, RMSE_vy_model, RMSE_r_model))
print("NN    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_NN, RMSE_vy_NN, RMSE_r_NN))
print('Highest Error in:')
print("Model - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_model, max_vy_model, max_r_model))
print("NN    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_NN, max_vy_NN, max_r_NN))


In [ ]:
dummy_x = np.array([[1, 1, 1, 1, 1],[2, 2, 2, 2, 2]])
dummy_x = np.expand_dims(dummy_x, axis=0)
dummy_y = model.predict(dummy_x)
print(dummy_y)